# Mode A: Smoke test

Checks that every stage runs on a real GPU with two tiny models (Qwen2.5-0.5B and 1.5B) on 24 questions. **Its numbers mean nothing and must never be reported.** Run this first after any change.

Runtime: about 10 to 20 minutes on a free T4, most of it installation.

## 1. Settings
Edit `REPO_URL` once the repository is on GitHub. Keep `USE_DRIVE = True` so results survive a disconnect.

In [ ]:
MODE = "smoke"
CONFIG = "configs/smoke.yaml"
REPO_URL = "https://github.com/YOUR-ACCOUNT/agreement-is-not-truth.git"  # or upload the ZIP to Drive
ZIP_ON_DRIVE = "/content/drive/MyDrive/agreement-is-not-truth.zip"     # used when REPO_URL is not set
USE_DRIVE = True
INSTALL_VLLM = False  # True on L4/A100 for speed; the code falls back to transformers if vLLM fails
PROJECT = "/content/agreement-is-not-truth"

## 2. Hardware check

In [ ]:
!nvidia-smi --query-gpu=name,memory.total --format=csv

## 3. Get the code and connect Drive

In [ ]:
import os, subprocess
if USE_DRIVE:
    from google.colab import drive
    drive.mount('/content/drive')
if not os.path.exists(PROJECT):
    if 'YOUR-ACCOUNT' not in REPO_URL:
        subprocess.run(['git', 'clone', REPO_URL, PROJECT], check=True)
    else:
        subprocess.run(['unzip', '-q', ZIP_ON_DRIVE, '-d', '/content'], check=True)
os.chdir(PROJECT)
if USE_DRIVE:
    store = '/content/drive/MyDrive/agreement-is-not-truth-results'
    os.makedirs(store, exist_ok=True)
    if not os.path.islink('results'):
        subprocess.run(['rm', '-rf', 'results'])
        os.symlink(store, 'results')
print(os.getcwd(), os.listdir('.'))

## 4. Install dependencies

In [ ]:
!pip install -q -e .[gpu,judge,dev]
if INSTALL_VLLM:
    !pip install -q vllm

## 5. Keys
Add `HF_TOKEN` and (optional) `OPENAI_API_KEY` in Colab's Secrets panel (key icon on the left). Without an OpenAI key the judge falls back to alias matching plus NLI, and the paper reports which judge was used.

In [ ]:
import os
os.environ['PYTORCH_CUDA_ALLOC_CONF'] = 'expandable_segments:True'  # less GPU memory fragmentation
try:
    from google.colab import userdata
    for name in ('HF_TOKEN', 'OPENAI_API_KEY'):
        try:
            os.environ[name] = userdata.get(name)
            print(name, 'loaded')
        except Exception:
            print(name, 'not set')
except ImportError:
    print('Not on Colab; set keys in the environment')

## 6. Offline tests
Runs the unit and integration tests on the simulated backend. They need no GPU and should all pass.

In [ ]:
!python -m pytest -q tests

## 7. Run the experiment
Each cell is one stage. Rerunning a cell skips finished work, so it is safe after a disconnect.

In [ ]:
!python -m pressure_probe.cli stage prepare --config $CONFIG

In [ ]:
!python -m pressure_probe.cli stage generate --config $CONFIG

In [ ]:
!python -m pressure_probe.cli stage judge --config $CONFIG

In [ ]:
!python -m pressure_probe.cli stage sample_scores --config $CONFIG

In [ ]:
!python -m pressure_probe.cli stage split --config $CONFIG

In [ ]:
!python -m pressure_probe.cli stage probe --config $CONFIG

In [ ]:
!python -m pressure_probe.cli stage metaqa --config $CONFIG

In [ ]:
!python -m pressure_probe.cli stage probe_scores --config $CONFIG

In [ ]:
!python -m pressure_probe.cli stage evaluate --config $CONFIG

## 9. Install LaTeX and pandoc (about 5 minutes)

In [ ]:
!apt-get -qq update && apt-get -qq install -y texlive-latex-recommended texlive-latex-extra texlive-fonts-recommended texlive-science latexmk > /dev/null && echo LaTeX done
# Colab's own pandoc is too old for the Word export, so install a current release.
!wget -q https://github.com/jgm/pandoc/releases/download/3.1.3/pandoc-3.1.3-1-amd64.deb -O /tmp/pandoc.deb && dpkg -i /tmp/pandoc.deb > /dev/null && pandoc --version | head -1

## 10. Write results into the paper
Fills tables, figures and numbers, compiles the IEEE PDF, writes the Word file, checks the page count and builds the ZIP.

In [ ]:
!python -m pressure_probe.cli report --config $CONFIG

In [ ]:
import json
report = json.load(open('paper/validation_report.json'))
print(json.dumps({k: v for k, v in report.items() if k != 'unresolved_tokens'}, indent=1))
print('Unresolved tokens:', report['unresolved_tokens'])

The smoke paper is for checking the pipeline only. Do not keep it.

## 11. Download

In [ ]:
from google.colab import files
files.download(f'dist/{MODE}_package.zip')
files.download('paper/paper.pdf')